In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

diff_threshold = 0.00005
diff_threshold_2 = 0.9
# Load the data into a pandas DataFrame
data = pd.read_csv('datasheet_csv.csv')  # Replace with the actual path to your data file
selected_columns = ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel','Magnitude_Gyro', 'Diff_Mag_Gyro', 'Diff_Mag_Accel'] 
data_1 = data[:600][selected_columns] # Replace with actual column names
data_2 = data[600:900][selected_columns] # Replace with actual column names
data_3 = data[900:1200][selected_columns] # Replace with actual column names
data_4 = data[1200:1500][selected_columns] # Replace with actual column names
# data['Magnitude_Accel'] = data['Magnitude_Accel'].abs()
    # Create new column: Is_High_Accel (1 if Magnitude_Accel > 0.5, 0 otherwise)
for i_d in [data_1, data_2, data_3, data_4]:
    if 'Magnitude_Accel' in data.columns:
        i_d['Is_High_Accel'] = np.where((i_d['Diff_Accel_X'] > diff_threshold) | (i_d['Diff_Accel_Y'] > diff_threshold) | (i_d['Diff_Accel_Z'] > diff_threshold), 1, 0)
        i_d['Is_High_Accel_2'] = np.where(i_d['Magnitude_Accel'] > diff_threshold_2, 1, 0)
        # print("\nCreated new column 'Is_High_Accel' based on Magnitude_Accel > 0.5.")
else:
    print("\nWarning: Magnitude_Accel column not found in the dataset.")
# Display the first few rows of the dataset
# print("First few rows of the dataset:")
# print(data.head())

# # Basic statistics of the dataset
print("\nBasic statistics of the dataset:")
print(data.describe())
print(f"Threshold for high acceleration: {diff_threshold}")
for i , data in enumerate([data_1, data_2, data_3, data_4], start=1):
    print(f"\n Phase {i} : Is_High_Accel value counts:")
    print(data['Is_High_Accel'].value_counts())

print(f"Threshold for high acceleration _ 2: {diff_threshold_2}")
for i , data in enumerate([data_1, data_2, data_3, data_4], start=1):
    print(f"\n Phase {i} : Is_High_Accel value counts:")
    print(data['Is_High_Accel_2'].value_counts())


Threshold for high acceleration _ 2: 0.9

 Phase 1 : Is_High_Accel value counts:
Is_High_Accel_2
1    546
0     54
Name: count, dtype: int64

 Phase 2 : Is_High_Accel value counts:
Is_High_Accel_2
1    299
0      1
Name: count, dtype: int64

 Phase 3 : Is_High_Accel value counts:
Is_High_Accel_2
0    286
1     14
Name: count, dtype: int64

 Phase 4 : Is_High_Accel value counts:
Is_High_Accel_2
1    285
0     15
Name: count, dtype: int64


In [3]:
import pandas as pd
import numpy as np

def compute_is_high_accel(df, diff_threshold, diff_threshold_2, diff_threshold_3):
    """
    Compute Is_High_Accel and Is_High_Accel_2 for a DataFrame given thresholds.
    """
    df = df.copy()  # Avoid modifying the original DataFrame
    df['Is_High_Accel'] = np.where(
        (df['Diff_Accel_X'] > diff_threshold) | (df['Diff_Accel_Y'] > diff_threshold) | (df['Diff_Accel_Z'] > diff_threshold),
        1, 0
    )
    df['Is_High_Accel_2'] = np.where(df['Magnitude_Accel'] > diff_threshold_2, 1, 0)
    df['Is_High_Accel_3'] = np.where(df['Diff_Mag_Accel'] > diff_threshold_3, 1, 0)
    return df

def score_thresholds(data_1, data_2, data_3, data_4, diff_threshold, diff_threshold_2, diff_threshold_3):
    """
    Compute a score based on the proportion of 1s in phases 1 and 4, and 0s in phases 2 and 3.
    """
    # Compute Is_High_Accel and Is_High_Accel_2 for each phase
    data_1 = compute_is_high_accel(data_1, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_2 = compute_is_high_accel(data_2, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_3 = compute_is_high_accel(data_3, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_4 = compute_is_high_accel(data_4, diff_threshold, diff_threshold_2, diff_threshold_3)
    
    # Calculate proportions
    prop_1_high = data_1['Is_High_Accel'].mean() + data_1['Is_High_Accel_2'].mean() + data_1['Is_High_Accel_3'].mean()
    prop_4_high = data_4['Is_High_Accel'].mean() + data_4['Is_High_Accel_2'].mean() + data_4['Is_High_Accel_3'].mean()
    prop_2_low = (1 - data_2['Is_High_Accel'].mean()) + (1 - data_2['Is_High_Accel_2'].mean()) + (1 - data_2['Is_High_Accel_3'].mean())
    prop_3_low = (1 - data_3['Is_High_Accel'].mean()) + (1 - data_3['Is_High_Accel_2'].mean()) + (1 - data_3['Is_High_Accel_3'].mean())
    
    # Total score: sum of desired proportions
    return prop_1_high + prop_4_high + prop_2_low + prop_3_low 

def find_optimal_thresholds(data_1, data_2, data_3, data_4):
    """
    Perform grid search to find optimal diff_threshold and diff_threshold_2.
    """
    # Define threshold ranges based on data ranges
    diff_thresholds = np.logspace(-5, -2, 20)  # e.g., 0.00001 to 0.01
    diff_thresholds_2 = np.linspace(0.5, 2.0, 20)  # e.g., 0.5 to 2.0
    diff_thresholds_3 = np.linspace(-5, -1, 20)  # e.g., 0.5 to 2.0
    
    best_score = -np.inf
    best_diff_threshold = None
    best_diff_threshold_2 = None
    best_diff_threshold_3 = None
    
    # Grid search
    for dt in diff_thresholds:
        for dt2 in diff_thresholds_2:
            for dt3 in diff_thresholds_3:
                score = score_thresholds(data_1, data_2, data_3, data_4, dt, dt2, dt3)
                if score > best_score:
                    best_score = score
                    best_diff_threshold = dt
                    best_diff_threshold_2 = dt2
                    best_diff_threshold_3 = dt3
    
    return best_diff_threshold, best_diff_threshold_2, best_diff_threshold_3, best_score

def main():
    # Load the data
    try:
        data = pd.read_csv('datasheet_csv.csv')
    except FileNotFoundError:
        print("Error: datasheet_csv.csv not found. Please provide the correct file path.")
        return
    
    # Check if dataset has enough rows
    if len(data) < 1500:
        print(f"Warning: Dataset has only {len(data)} rows, expected at least 1500.")
    
    # Select columns
    selected_columns = ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel',
                       'Magnitude_Gyro', 'Diff_Mag_Gyro', 'Diff_Mag_Accel']
    
    # Ensure columns exist
    selected_columns = [col for col in selected_columns if col in data.columns]
    if not all(col in data.columns for col in ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel']):
        missing = [col for col in ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel'] if col not in data.columns]
        print(f"Error: Missing columns {missing}.")
        return
    
    # Split into phases
    data_1 = data[:600][selected_columns]
    data_2 = data[600:900][selected_columns]
    data_3 = data[900:1200][selected_columns]
    data_4 = data[1200:1500][selected_columns]
    
    # Find optimal thresholds
    print("\nSearching for optimal thresholds...")
    best_diff_threshold, best_diff_threshold_2, best_diff_threshold_3 ,best_score = find_optimal_thresholds(data_1, data_2, data_3, data_4)
    
    print(f"\nOptimal diff_threshold: {best_diff_threshold:.6f}")
    print(f"Optimal diff_threshold_2: {best_diff_threshold_2:.6f}")
    print(f"Optimal diff_threshold_3: {best_diff_threshold_3:.6f}")
    print(f"Best score: {best_score:.6f}")
    
    # Apply optimal thresholds
    datasets = [data_1, data_2, data_3, data_4]
    for i, d in enumerate(datasets, 1):
        d['Is_High_Accel'] = np.where(
            (d['Diff_Accel_X'] > best_diff_threshold) | (d['Diff_Accel_Y'] > best_diff_threshold) | (d['Diff_Accel_Z'] > best_diff_threshold),
            1, 0
        )
        d['Is_High_Accel_2'] = np.where(d['Magnitude_Accel'] > best_diff_threshold_2, 1, 0)
        d['Is_High_Accel_3'] = np.where(d['Diff_Mag_Accel'] > best_diff_threshold_3, 1, 0)
    
    # Basic statistics
    print("\nBasic statistics of the dataset:")
    print(data[selected_columns].describe())
    
    # Print value counts for each phase
    print(f"\nThreshold for Is_High_Accel: {best_diff_threshold:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel value counts:")
        print(d['Is_High_Accel'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))
    
    print(f"\nThreshold for Is_High_Accel_2: {best_diff_threshold_2:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel_2 value counts:")
        print(d['Is_High_Accel_2'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))

    print(f"\nThreshold for Is_High_Accel_3: {best_diff_threshold_3:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel_3 value counts:")
        print(d['Is_High_Accel_3'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))
    
    # Save modified datasets
    for i, d in enumerate(datasets, 1):
        d.to_csv(f'dataset_phase_{i}_with_is_high_accel.csv', index=False)
        print(f"\nPhase {i} dataset saved to 'dataset_phase_{i}_with_is_high_accel.csv'")

if __name__ == "__main__":
    main()


Searching for optimal thresholds...

Optimal diff_threshold: 0.000264
Optimal diff_threshold_2: 0.973684
Optimal diff_threshold_3: -5.000000
Best score: 8.850000

Basic statistics of the dataset:
       Diff_Accel_X  Diff_Accel_Y  Diff_Accel_Z  Magnitude_Accel  \
count   1499.000000   1499.000000    1499.00000      1500.000000   
mean       0.004267      0.003470       0.00087         0.967702   
std        0.008194      0.005948       0.00180         0.529288   
min        0.000000      0.000000       0.00000        -0.862002   
25%        0.000050      0.000050       0.00005         0.932169   
50%        0.000250      0.000400       0.00015         0.963536   
75%        0.005050      0.004525       0.00095         1.103276   
max        0.077600      0.044300       0.02500         5.233343   

       Magnitude_Gyro  Diff_Mag_Gyro  Diff_Mag_Accel  
count     1500.000000    1499.000000     1499.000000  
mean         0.531973       0.362153        0.191070  
std          0.842216    

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compute_is_high_accel(df, diff_threshold=None, diff_threshold_2=None, diff_threshold_3=None):
    """
    Compute Is_High_Accel and/or Is_High_Accel_2 for a DataFrame given thresholds.
    """
    df = df.copy()  # Avoid modifying the original DataFrame
    if diff_threshold is not None:
        df['Is_High_Accel'] = np.where(
            (df['Diff_Accel_X'] > diff_threshold) | (df['Diff_Accel_Y'] > diff_threshold) | (df['Diff_Accel_Z'] > diff_threshold),
            1, 0
        )
    if diff_threshold_2 is not None:
        df['Is_High_Accel_2'] = np.where(df['Magnitude_Accel'] > diff_threshold_2, 1, 0)

    if diff_threshold_3 is not None:
        df['Is_High_Accel_3'] = np.where(df['Diff_Mag_Accel'] > diff_threshold_3, 1, 0)
    return df

def score_threshold(data_1, data_2, data_3, data_4, diff_threshold=None, diff_threshold_2=None, diff_threshold_3=None):
    """
    Compute a score based on the proportion of 1s in phases 1 and 4, and 0s in phases 2 and 3.
    """
    # Compute Is_High_Accel or Is_High_Accel_2
    data_1 = compute_is_high_accel(data_1, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_2 = compute_is_high_accel(data_2, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_3 = compute_is_high_accel(data_3, diff_threshold, diff_threshold_2, diff_threshold_3)
    data_4 = compute_is_high_accel(data_4, diff_threshold, diff_threshold_2, diff_threshold_3)
    
    # Select the column to score
    col = 'Is_High_Accel' if diff_threshold is not None else 'Is_High_Accel_2' if diff_threshold_2 is not None else 'Is_High_Accel_3'
    
    # Calculate proportions
    prop_1_high = data_1[col].mean()
    prop_4_high = data_4[col].mean()
    prop_2_low = 1 - data_2[col].mean()
    prop_3_low = 1 - data_3[col].mean()
    
    # Total score
    return prop_1_high + prop_4_high + prop_2_low + prop_3_low

def find_optimal_thresholds(data_1, data_2, data_3, data_4):
    """
    Find optimal diff_threshold and diff_threshold_2 separately.
    """
    # Threshold ranges
    diff_thresholds = np.logspace(-5, -2, 50)  # 0.00001 to 0.01 for Diff_Accel_X/Y/Z
    diff_thresholds_2 = np.linspace(0.5, 2.0, 50)  # 0.5 to 2.0 for Magnitude_Accel
    diff_thresholds_3 = np.logspace(-5, -1, 50)  # -5 to -1 for Diff_Mag_Accel
    
    # Optimize diff_threshold
    best_score_1 = -np.inf
    best_diff_threshold = None
    scores_1 = []
    for dt in diff_thresholds:
        score = score_threshold(data_1, data_2, data_3, data_4, diff_threshold=dt)
        scores_1.append((dt, score))
        if score > best_score_1:
            best_score_1 = score
            best_diff_threshold = dt
    
    # Optimize diff_threshold_2
    best_score_2 = -np.inf
    best_diff_threshold_2 = None
    scores_2 = []
    for dt2 in diff_thresholds_2:
        score = score_threshold(data_1, data_2, data_3, data_4, diff_threshold_2=dt2)
        scores_2.append((dt2, score))
        if score > best_score_2:
            best_score_2 = score
            best_diff_threshold_2 = dt2
        
    # Optimize diff_threshold_3
    best_score_3 = -np.inf
    best_diff_threshold_3 = None
    scores_3 = []
    for dt3 in diff_thresholds_3:
        score = score_threshold(data_1, data_2, data_3, data_4, diff_threshold_3=dt3)
        scores_3.append((dt3, score))
        if score > best_score_3:
            best_score_3 = score
            best_diff_threshold_3 = dt3
    
    return (best_diff_threshold, best_score_1, scores_1), (best_diff_threshold_2, best_score_2, scores_2), (best_diff_threshold_3, best_score_3, scores_3)

def main():
    # Load the data
    try:
        data = pd.read_csv('datasheet_csv.csv')
    except FileNotFoundError:
        print("Error: datasheet_csv.csv not found. Please provide the correct file path.")
        return
    
    # Check if dataset has enough rows
    if len(data) < 1500:
        print(f"Warning: Dataset has only {len(data)} rows, expected at least 1500.")
    
    # Select columns
    selected_columns = ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel',
                       'Magnitude_Gyro', 'Diff_Mag_Gyro', 'Diff_Mag_Accel']
    
    # Ensure columns exist
    selected_columns = [col for col in selected_columns if col in data.columns]
    required_columns = ['Diff_Accel_X', 'Diff_Accel_Y', 'Diff_Accel_Z', 'Magnitude_Accel']
    if not all(col in data.columns for col in required_columns):
        missing = [col for col in required_columns if col not in data.columns]
        print(f"Error: Missing columns {missing}.")
        return
    
    # Split into phases
    data_1 = data[:600][selected_columns]
    data_2 = data[600:900][selected_columns]
    data_3 = data[900:1200][selected_columns]
    data_4 = data[1200:1500][selected_columns]
    
    # Find optimal thresholds
    print("\nSearching for optimal thresholds...")
    (best_diff_threshold, best_score_1, scores_1), (best_diff_threshold_2, best_score_2, scores_2), (best_diff_threshold_3, best_score_3, scores_3) = find_optimal_thresholds(data_1, data_2, data_3, data_4)
    
    print(f"\nOptimal diff_threshold (for Diff_Accel_X/Y/Z): {best_diff_threshold:.6f}")
    print(f"Best score (Is_High_Accel): {best_score_1:.6f}")
    print(f"Optimal diff_threshold_2 (for Magnitude_Accel): {best_diff_threshold_2:.6f}")
    print(f"Best score (Is_High_Accel_2): {best_score_2:.6f}")
    print(f"Optimal diff_threshold_3 (for Diff_Mag_Accel): {best_diff_threshold_3:.6f}")
    print(f"Best score (Is_High_Accel_3): {best_score_3:.6f}")
    
    # Apply optimal thresholds
    datasets = [data_1, data_2, data_3, data_4]
    for i, d in enumerate(datasets, 1):
        d['Is_High_Accel'] = np.where(
            (d['Diff_Accel_X'] > best_diff_threshold) | (d['Diff_Accel_Y'] > best_diff_threshold) | (d['Diff_Accel_Z'] > best_diff_threshold),
            1, 0
        )
        d['Is_High_Accel_2'] = np.where(d['Magnitude_Accel'] > best_diff_threshold_2, 1, 0)
        d['Is_High_Accel_3'] = np.where(d['Diff_Mag_Accel'] > best_diff_threshold_3, 1, 0)
    
    # Basic statistics
    print("\nBasic statistics of the dataset:")
    print(data[selected_columns].describe())
    
    # Print value counts for Is_High_Accel
    print(f"\nThreshold for Is_High_Accel: {best_diff_threshold:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel value counts:")
        print(d['Is_High_Accel'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))
    
    # Print value counts for Is_High_Accel_2
    print(f"\nThreshold for Is_High_Accel_2: {best_diff_threshold_2:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel_2 value counts:")
        print(d['Is_High_Accel_2'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))
        
    # Print value counts for Is_High_Accel_3
    print(f"\nThreshold for Is_High_Accel_3: {best_diff_threshold_3:.6f}")
    for i, d in enumerate(datasets, 1):
        print(f"\nPhase {i} : Is_High_Accel_3 value counts:")
        print(d['Is_High_Accel_3'].value_counts().rename({1: 'High Accel (1)', 0: 'Low Accel (0)'}))
    # Plot score vs. thresholds
    # Plot for diff_threshold
    thresholds_1, scores_1 = zip(*scores_1)
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds_1, scores_1, marker='o')
    plt.xscale('log')
    plt.xlabel('Diff Threshold (Diff_Accel_X/Y/Z)')
    plt.ylabel('Score')
    plt.title('Score vs. Diff Threshold for Is_High_Accel')
    plt.grid(True)
    plt.savefig('threshold_score_is_high_accel.png')
    plt.close()
    
    # Plot for diff_threshold_2
    thresholds_2, scores_2 = zip(*scores_2)
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds_2, scores_2, marker='o')
    plt.xlabel('Diff Threshold 2 (Magnitude_Accel)')
    plt.ylabel('Score')
    plt.title('Score vs. Diff Threshold 2 for Is_High_Accel_2')
    plt.grid(True)
    plt.savefig('threshold_score_is_high_accel_2.png')
    plt.close()
    print("\nScore vs. threshold plots saved to 'threshold_score_is_high_accel.png' and 'threshold_score_is_high_accel_2.png'")

    # Plot for diff_threshold_3
    thresholds_3, scores_3 = zip(*scores_3)
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds_3, scores_3, marker='o')
    plt.xlabel('Diff Threshold 3 (Diff_Mag_Accel)')
    plt.ylabel('Score')
    plt.title('Score vs. Diff Threshold 3 for Is_High_Accel_3')
    plt.grid(True)
    plt.savefig('threshold_score_is_high_accel_3.png')
    plt.close()
    print("\nScore vs. threshold plots saved to 'threshold_score_is_high_accel_3.png'")
    
    # Save modified datasets
    for i, d in enumerate(datasets, 1):
        d.to_csv(f'dataset_phase_{i}_with_is_high_accel.csv', index=False)
        print(f"\nPhase {i} dataset saved to 'dataset_phase_{i}_with_is_high_accel.csv'")

if __name__ == "__main__":
    main()


Searching for optimal thresholds...

Optimal diff_threshold (for Diff_Accel_X/Y/Z): 0.000256
Best score (Is_High_Accel): 3.490000
Optimal diff_threshold_2 (for Magnitude_Accel): 0.959184
Best score (Is_High_Accel_2): 3.238333
Optimal diff_threshold_3 (for Diff_Mag_Accel): 0.022230
Best score (Is_High_Accel_3): 3.241667

Basic statistics of the dataset:
       Diff_Accel_X  Diff_Accel_Y  Diff_Accel_Z  Magnitude_Accel  \
count   1499.000000   1499.000000    1499.00000      1500.000000   
mean       0.004267      0.003470       0.00087         0.967702   
std        0.008194      0.005948       0.00180         0.529288   
min        0.000000      0.000000       0.00000        -0.862002   
25%        0.000050      0.000050       0.00005         0.932169   
50%        0.000250      0.000400       0.00015         0.963536   
75%        0.005050      0.004525       0.00095         1.103276   
max        0.077600      0.044300       0.02500         5.233343   

       Magnitude_Gyro  Diff_Mag